# Reflection Design Pattern — Lab 2: SQL Reflection

An agent that writes a SQL query, runs it, then reflects on the query and its
result and produces a better one. Same reflection pattern as lab-1, but the
feedback signal is the query result (or SQL error), not an image.

Logic lives in `src/`; this notebook wires it together.


In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv

from src import config
from src.db import get_schema, execute_sql
from src.rendering import print_html
from src.agent import build_agent

load_dotenv()

from langfuse import get_client
from langfuse.langchain import CallbackHandler

langfuse = get_client()
langfuse_handler = CallbackHandler()


In [ ]:
print(get_schema(str(config.DB_PATH)))


In [ ]:
agent = build_agent()


In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": config.INSTRUCTION}]},
    config={"configurable": {"thread_id": "sql-reflection-1"}, "callbacks": [langfuse_handler]},
)
print(result["messages"][-1].content)
langfuse.flush()


In [ ]:
from langchain_core.messages import ToolMessage

for m in result["messages"]:
    if isinstance(m, ToolMessage):
        print_html(m.content, title=f"Tool output: {m.name}")
